In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [9]:
df = pd.read_pickle('data/df.pkl')

In [10]:
df

,year,plot_id,ndvi_peak,ndvi_peak_doy,ndvi_min,ndvi_integral,ndvi_sos,ndvi_eos,ndvi_los,ndvi_greenup_slope,...,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_x,slope_y,slope_squared,slope_log,local_relief,total_relief_log
0,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
1,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
2,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
3,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
4,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161396,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969
161397,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969
161398,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969
161399,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969


In [12]:
non_features = [
    col for col in df.columns if 'ndvi' in col
] + [
    'year', 'plot_id', 'geometry', 'key_0', 'area_m2'
]
features = [col for col in df.columns if col not in non_features]
target = np.log1p(df['ndvi_integral'])

TypeError: argument of type 'int' is not iterable

In [8]:
# df

# Copy original features
# X_eng = X.copy()

# Add squared features for the inverted/weak U shapes
for f in ['total_relief', 'local_relief', 'slope_log']:
    df[f + '_sq'] = df[f]**2
    
    
features = [col for col in df.columns if col not in non_features]
X = df[features]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

TypeError: argument of type 'int' is not iterable

In [ ]:
# top_features = ['total_relief','diurnal_temp_range','aspect_max_cos','slope_log','area_ha','local_relief']

# for f in top_features:
#     x = df[f]
#     plt.scatter(x, target)
#     plt.title(f)
#     plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute correlations only for features vs target
corr_with_target = df[features + ["ndvi_integral"]].corr()["ndvi_integral"].drop("ndvi_integral")
corr_with_target = corr_with_target.sort_values(ascending=False)

# Plot barplot of correlations
plt.figure(figsize=(10,10))
sns.barplot(x=corr_with_target.values, y=corr_with_target.index, palette="coolwarm")
plt.title("Feature Correlation with NDVI Integral", fontsize=16)
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.show()


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# Split your data
# First split off the test set (say 20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, target, test_size=0.2, random_state=42
)

# Then split the remaining into train + validation (say 20% of remaining)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

# Initialize Ridge model with all key parameters exposed
ridge_model = Ridge(
    alpha=1.0,               # Regularization strength
    fit_intercept=True,      # Whether to calculate the intercept for this model
    # normalize=False,         # Deprecated if using StandardScaler (already scaled)
    copy_X=True,             # Whether to copy X or overwrite
    max_iter=None,           # Maximum iterations for solver to converge
    tol=1e-3,                # Tolerance for convergence
    solver='auto',           # Solver choice: 'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'
    random_state=42          # Only used for 'sag', 'saga'
)

# Fit the model
ridge_model.fit(X_train, y_train)

# Predictions
y_pred = ridge_model.predict(X_test)

# Evaluation
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"Test R²: {r2:.4f}")
print(f"Test RMSE: {rmse:.4f}")


In [ ]:
param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'],
    'max_iter': [None, 500, 1000, 2000],
    'tol': [1e-4, 1e-3, 1e-2]
}

grid = GridSearchCV(Ridge(fit_intercept=True, copy_X=True), param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

# Evaluate best model
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
print("Test R² (best model):", r2_score(y_test, y_pred_best))


In [4]:
plt.scatter(y_test, y_pred_best)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--')
plt.plot

NameError: name 'plt' is not defined

In [5]:
import matplotlib.pyplot as plt

residuals = y_test - y_pred_best
plt.scatter(y_test, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("observed NDVI Integral")
plt.ylabel("Residuals")
plt.title("Ridge Residuals vs Predicted")
plt.show()


NameError: name 'y_test' is not defined

In [6]:
# coef = pd.Series(best_model.coef_, index=features)
# print(coef.sort_values())


In [23]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd

In [24]:
# # Split data
# X_train, X_test, y_train, y_test = train_test_split(
#     X_scaled, target, test_size=0.2, random_state=42
# )

In [25]:
# Lasso with cross-validated alpha (regularization strength)
lasso = LassoCV(
    alphas=np.logspace(-4, 1, 50),  # range of alpha to try
    cv=5,
    max_iter=10000,
    tol=1e-4,
    random_state=42
)
lasso.fit(X_train, y_train)

LassoCV(alphas=array([1.00000000e-04, 1.26485522e-04, 1.59985872e-04, 2.02358965e-04,
       2.55954792e-04, 3.23745754e-04, 4.09491506e-04, 5.17947468e-04,
       6.55128557e-04, 8.28642773e-04, 1.04811313e-03, 1.32571137e-03,
       1.67683294e-03, 2.12095089e-03, 2.68269580e-03, 3.39322177e-03,
       4.29193426e-03, 5.42867544e-03, 6.86648845e-03, 8.68511374e-03,
       1.09854114e-02, 1.38949549e-0...
       7.19685673e-02, 9.10298178e-02, 1.15139540e-01, 1.45634848e-01,
       1.84206997e-01, 2.32995181e-01, 2.94705170e-01, 3.72759372e-01,
       4.71486636e-01, 5.96362332e-01, 7.54312006e-01, 9.54095476e-01,
       1.20679264e+00, 1.52641797e+00, 1.93069773e+00, 2.44205309e+00,
       3.08884360e+00, 3.90693994e+00, 4.94171336e+00, 6.25055193e+00,
       7.90604321e+00, 1.00000000e+01]),
        cv=5, max_iter=10000, random_state=42)

In [28]:
# Predictions
y_pred = lasso.predict(X_test)

# Evaluation
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Best alpha: {lasso.alpha_:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# Feature selection
coef = pd.Series(lasso.coef_, index=features)
selected_features = coef[coef != 0].sort_values()
print("Selected features by Lasso:")
print(selected_features)

# Optional: plot coefficient magnitudes
plt.figure(figsize=(10,6))
selected_features.plot(kind="barh")
plt.xlabel("Coefficient magnitude")
plt.ylabel("Feature")
plt.title("Lasso Selected Feature Coefficients")
plt.show()

Best alpha: 0.0007
Test R²: 0.4844
Test RMSE: 0.1056


ValueError: Length of passed values is 64, index implies 61.

In [ ]:
residuals = (y_test) - lasso.predict(X_test)
plt.scatter(y_test, lasso.predict(X_test))
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Predicted log(NDVI)")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted (Log-transformed Lasso)")
plt.show()


In [ ]:
residuals = (y_test) - lasso.predict(X_test)
plt.scatter(y_test, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted log(NDVI)")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted (Log-transformed Lasso)")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

# --- 1. Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# --- 2. Identify top curved features ---
curved_features = ['total_relief', 'local_relief', 'slope_log']

# --- 3. Create squared terms ---
for f in curved_features:
    X_scaled[f+'_sq'] = X_scaled[f]**2

# --- 4. Create pairwise interactions among curved features ---
from itertools import combinations
for f1, f2 in combinations(curved_features, 2):
    X_scaled[f'{f1}_x_{f2}'] = X_scaled[f1] * X_scaled[f2]

# --- 5. Split data ---
y_log = np.log(target)  # log-transform target
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_log, test_size=0.2, random_state=42)

# --- 6. Fit LassoCV ---
lasso = LassoCV(alphas=np.logspace(-4,1,50), cv=5, max_iter=20000, random_state=42)
lasso.fit(X_train, y_train)

# --- 7. Evaluate ---
y_pred = lasso.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Best alpha: {lasso.alpha_:.6f}")
print(f"Test R²: {r2:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# --- 8. Feature selection ---
coef = pd.Series(lasso.coef_, index=X_scaled.columns)
selected_features = coef[coef != 0].sort_values(ascending=False)
print("Selected features by Lasso:")
print(selected_features)

# --- 9. Plot coefficients ---
plt.figure(figsize=(10,8))
selected_features.plot(kind='barh')
plt.title("Lasso Selected Features (Squares + Interactions)")
plt.xlabel("Coefficient magnitude")
plt.show()

# --- 10. Residuals ---
residuals = y_test - y_pred
plt.figure(figsize=(8,6))
plt.scatter(y_test, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted log(NDVI)")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted (with Squares + Interactions)")
plt.show()
